In [2]:
!pip install -q "transformers>=4.46.0" "accelerate>=0.26.0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [3]:
import threading, time
from transformers import TextIteratorStreamer


def prompt_of_len(n_tokens: int) -> str:
  base = "Explain the following in detail.\n"
  filler = "A data center serves many inference requests at once. " * 600
  ids = tok(base + filler)["input_ids"][:n_tokens]
  return tok.decode(ids)


def measure_stream(prompt: str, new_tokens: int = 128):
  enc = tok(prompt, return_tensors="pt").to("cuda")
  streamer = TextIteratorStreamer(
      tok, skip_prompt=True, skip_special_tokens=True
  )
  kwargs = dict(
      **enc, max_new_tokens=new_tokens, do_sample=False, streamer=streamer
  )
  th = threading.Thread(target=model.generate, kwargs=kwargs)
  t0 = time.time()
  th.start()
  stamps = []
  for _ in streamer:
    stamps.append(time.time())
  th.join()
  ttft = stamps[0] - t0
  # mean inter-token gap over the tokens after the first
  if len(stamps) > 1:
    gaps = [b - a for a, b in zip(stamps, stamps[1:])]
    tpot = sum(gaps) / len(gaps)
  else:
    tpot = 0.0
  total = stamps[-1] - t0
  return {
      "ttft_s": round(ttft, 4),
      "tpot_s": round(tpot, 4),
      "total_s": round(total, 4),
      "n_tokens": len(stamps),
  }


# Warm-up لتفادي احتساب زمن تهيئة الـ CUDA context في القياسات
measure_stream(prompt_of_len(128), new_tokens=8)

ttft_by_len = {}
for n in [128, 512, 2048]:
  r = measure_stream(prompt_of_len(n))
  ttft_by_len[str(n)] = r["ttft_s"]
  print(n, r)

128 {'ttft_s': 0.108, 'tpot_s': 0.1172, 'total_s': 15.1066, 'n_tokens': 129}
512 {'ttft_s': 0.1749, 'tpot_s': 0.1449, 'total_s': 18.7256, 'n_tokens': 129}
2048 {'ttft_s': 0.7476, 'tpot_s': 0.0683, 'total_s': 9.4961, 'n_tokens': 129}


In [5]:
import gc, json


def kv_formula_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
  return 2 * layers * kv_heads * head_dim * dbytes / 1024  # 28.0 KB


def cache_bytes(pkv):
  """Bytes the KV cache itself holds, read straight off the cache tensors."""
  if pkv is None:
    return 0
  if hasattr(pkv, "key_cache"):  # transformers returns a Cache object
    tensors = list(pkv.key_cache) + list(pkv.value_cache)
  else:  # legacy tuple of (k, v) per layer
    tensors = [t for layer in pkv for t in layer]
  # Filter out None values before attempting to call .numel()
  tensors = [t for t in tensors if t is not None]
  if not tensors:
    return 0
  return sum(t.numel() * t.element_size() for t in tensors)


def measure_kv(context: int, new_tokens: int = 256):
  torch.cuda.empty_cache()
  gc.collect()
  torch.cuda.reset_peak_memory_stats()
  enc = tok(prompt_of_len(context), return_tensors="pt").to("cuda")
  before = torch.cuda.memory_allocated()
  out = model.generate(
      **enc,
      max_new_tokens=new_tokens,
      do_sample=False,
      use_cache=True,
      return_dict_in_generate=True,
  )
  torch.cuda.synchronize()
  peak = torch.cuda.max_memory_allocated()
  total_tokens = out.sequences.shape[1]  # prompt + generated, the full KV span
  return {
      "context": context,
      "total_tokens": int(total_tokens),
      # what the whole generation cost, cache and activations together
      "peak_kb_per_token": round((peak - before) / total_tokens / 1024, 1),
      # the cache on its own, which is what the formula predicts
      "kv_kb_per_token": round(
          cache_bytes(out.past_key_values) / total_tokens / 1024, 1
      ),
  }


formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)
kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
  print(r, "  vs formula", formula, "KB/token")

with open("kv_check.json", "w") as f:
  json.dump(
      {
          "formula_kb_per_token": formula,
          "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
          "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"],
      },
      f,
  )


formula KB/token: 28.0
{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 81.7, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 258.2, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 484.3, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token


In [6]:
# 24 requests: 18 that want 32 tokens, 6 that want 256. 2112 useful tokens.
QUEUE = [32, 32, 32, 256] * 6


def static_queue(
    batch: int, prompt: str = "Explain what an inference server does."
):
  t0 = time.time()
  useful = 0
  slots = 0
  for i in range(0, len(QUEUE), batch):
    chunk = QUEUE[i : i + batch]
    n = max(chunk)  # the batch runs until the slowest
    enc = tok(
        [prompt] * len(chunk), return_tensors="pt", padding=True
    ).to("cuda")
    model.generate(**enc, max_new_tokens=n, do_sample=False)
    useful += sum(chunk)  # tokens anyone actually asked for
    slots += n * len(chunk)  # token-slots the GPU actually decoded
  dt = time.time() - t0
  return {
      "batch": batch,
      "wall_s": round(dt, 2),
      "tokens_per_s": round(useful / dt, 1),
      "slot_efficiency": round(useful / slots, 3),
  }


batch_rows = {}
for n in [1, 4, 8]:
  r = static_queue(n)
  batch_rows[str(n)] = r["tokens_per_s"]
  print(r)

{'batch': 1, 'wall_s': 74.87, 'tokens_per_s': 28.2, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 50.73, 'tokens_per_s': 41.6, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 26.03, 'tokens_per_s': 81.1, 'slot_efficiency': 0.344}


In [7]:
import json
from google.colab import files

baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}

with open("baselines.json", "w") as f:
  json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

# تنزيل الملف مباشرة
files.download("baselines.json")

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.108,
    "512": 0.1749,
    "2048": 0.7476
  },
  "tpot_s": 0.0609,
  "batch": {
    "1": 28.2,
    "4": 41.6,
    "8": 81.1
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
with open("baselines.json", "r") as f:
  b = json.load(f)
with open("kv_check.json", "r") as f:
  k = json.load(f)

# 1. فحص البنية والمفاتيح الأساسية
assert all(k_key in b for k_key in ["model", "dtype", "ttft_s", "tpot_s", "batch"])
assert "128" in b["ttft_s"] and "2048" in b["ttft_s"]

# 2. التحقق من أن زمن أول توكن يرتفع مع طول السياق
assert b["ttft_s"]["2048"] > b["ttft_s"]["128"]

# 3. التحقق من أن إنتاجية الدفعة 8 تتفوق على الدفعة 1
assert float(b["batch"]["8"]) > float(b["batch"]["1"])

# 4. التحقق من تطابق حجم الـ KV Cache مع المعادلة الحسابية (28.0 KB)
assert abs(k["measured_kb_per_token"] - 28.0) < 0.1

print("=" * 30)
print("GREEN CHECK: PASS")
print("=" * 30)

GREEN CHECK: PASS
